In [2]:
import boto3, botocore
from botocore.exceptions import ClientError
import os, time, json, io, zipfile
from datetime import date
from dotenv import load_dotenv


from misc import load_from_yaml, save_to_yaml
# import iam, s3, lf, rds, vpc, ec2

load_dotenv(".env")
boto3.setup_default_session(profile_name="AMShah")

In [ ]:
ALL_IN_ONE_SG = ''
ACCOUNT_ID        = os.environ['AWS_ACCOUNT_ID_ROOT']
REGION            = os.environ['AWS_DEFAULT_REGION']
VPC_ID            = os.environ['AWS_DEFAULT_VPC']
SECURITY_GROUP_ID = os.environ['AWS_DEFAULT_SG_ID']
SUBNET_IDS        = SUBNET_IDS = os.environ["AWS_DEFAULT_SUBNET_IDS"].split(":")
SUBNET_ID         = SUBNET_IDS[0]
AWS_INSTANCE_ID_JMASTER   = os.environ['AWS_INSTANCE_ID_JMASTER']
# AWS_INSTANCE_ID_JAGENT    = os.environ['AWS_INSTANCE_ID_JAGENT']
AWS_DEFAULT_IMAGE_ID      = os.environ['AWS_DEFAULT_IMAGE_ID']
AWS_DEFAULT_KEY_PAIR_NAME = os.environ['AWS_DEFAULT_KEY_PAIR_NAME']
AWS_DEFAULT_INSTANCE_TYPE = os.environ['AWS_DEFAULT_INSTANCE_TYPE']

In [5]:
sts_client           = boto3.client('sts')
rds_client           = boto3.client('rds')
iam_client           = boto3.client('iam')
s3_client            = boto3.client('s3')
glue_client          = boto3.client('glue')
lakeformation_client = boto3.client('lakeformation')
stepfunctions_client = boto3.client('stepfunctions')
apigateway_client    = boto3.client('apigateway')
lsn_client           = boto3.client('lambda')
events_client        = boto3.client('events')
sqs_client           = boto3.client('sqs')

emr_client = boto3.client('emr', region_name=REGION)

In [ ]:
ec2_client           = boto3.client('ec2', region_name=REGION)
ec2_resource         = boto3.resource('ec2', region_name=REGION)

# # Example: Get a specific VPC
# vpc = ec2_resource.Vpc('vpc_id')

# # Example: Get a specific EBS volume
# volume = ec2_resource.Volume('volume_id')

### STS (Security Token Service)

- [Calm Cloud Security - AWS STS Assume Theory - What is STS and Why?](https://www.youtube.com/watch?v=N0FvntzTigg)
- [Calm Cloud Security - AWS STS Assume with AWS CLI and Terraform](https://www.youtube.com/watch?v=9cQWcJueyMQ&t=1s)
- [How to use AWS Secure Token Service to manage credentials](https://www.youtube.com/watch?v=grW4Wh28LFA&t=302s)
- [How to assume a role with AWS Security Token Service (STS)](https://www.youtube.com/watch?v=dqF4VJCska4&t=751s)

#### IAM User

In [ ]:
# Configuration
group_name = "SecureDataEngineers"
user_name = "john.doe"
path = "/engineering/"  # Optional path
password = "Strong#Password123!"
permissions_boundary = "arn:aws:iam::aws:policy/ReadOnlyAccess"
managed_policy_arns = [
    "arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess",
    "arn:aws:iam::aws:policy/CloudWatchReadOnlyAccess",
]
inline_policy_name = "CustomS3RestrictionPolicy"
inline_policy_document = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Deny", "Action": "s3:DeleteObject", "Resource": "*"}],
}
tags = [
    {"Key": "Environment", "Value": "Dev"},
    {"Key": "Department", "Value": "Security"},
]

#### Not Tested

In [ ]:

def create_group(group_name, path="/"):
    try:
        iam_client.create_group(GroupName=group_name, Path=path)
        print(f"✅ Group created: {group_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            print(f"⚠️ Group already exists: {group_name}")
        else:
            raise


def create_user(user_name, path="/", tags=None, permissions_boundary=None):
    try:
        iam_client.create_user(
            UserName=user_name,
            Path=path,
            Tags=tags or [],
            PermissionsBoundary=permissions_boundary,
        )
        print(f"✅ User created: {user_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            print(f"⚠️ User already exists: {user_name}")
        else:
            raise


def create_login_profile(user_name, password, password_reset_required=True):
    try:
        iam_client.create_login_profile(
            UserName=user_name,
            Password=password,
            PasswordResetRequired=password_reset_required,
        )
        print(f"🔐 Console login profile created for user: {user_name}")
    except ClientError as e:
        print(f"❌ Error creating login profile: {e}")


def create_access_keys(user_name):
    try:
        response = iam_client.create_access_key(UserName=user_name)
        access_key = response["AccessKey"]
        print("🔑 Access Key created:")
        print(f"AccessKeyId: {access_key['AccessKeyId']}")
        print(f"SecretAccessKey: {access_key['SecretAccessKey']}")
    except ClientError as e:
        print(f"❌ Error creating access keys: {e}")


def add_user_to_group(user_name, group_name):
    try:
        iam_client.add_user_to_group(UserName=user_name, GroupName=group_name)
        print(f"👥 User {user_name} added to group {group_name}")
    except ClientError as e:
        print(f"❌ Error adding user to group: {e}")


def attach_managed_policies(group_name, policy_arns):
    for policy_arn in policy_arns:
        try:
            iam_client.attach_group_policy(GroupName=group_name, PolicyArn=policy_arn)
            print(f"📎 Attached managed policy to group: {policy_arn}")
        except ClientError as e:
            print(f"❌ Error attaching policy: {e}")


def put_inline_policy(user_name, policy_name, policy_document):
    try:
        iam_client.put_user_policy(
            UserName=user_name,
            PolicyName=policy_name,
            PolicyDocument=json.dumps(policy_document),
        )
        print(f"📄 Inline policy '{policy_name}' attached to user {user_name}")
    except ClientError as e:
        print(f"❌ Error attaching inline policy: {e}")


create_group(group_name, path)
create_user(user_name, path, tags, permissions_boundary)
create_login_profile(user_name, password)
create_access_keys(user_name)
add_user_to_group(user_name, group_name)
attach_managed_policies(group_name, managed_policy_arns)
put_inline_policy(user_name, inline_policy_name, inline_policy_document)


In [ ]:
# Step 1: Create IAM User
user_name = 'test-mfa-user'
iam_client.create_user(UserName=user_name)
print(f"IAM User '{user_name}' created.")

# Optional: Step 2: Add a login profile (required for console login)
iam_client.create_login_profile(
    UserName=user_name,
    Password='StrongP@ssword123!',  # Must meet password policy
    PasswordResetRequired=True
)